In [0]:
source_table = dbutils.widgets.get("source_table")
rs_table = dbutils.widgets.get("rs_table")
rs_staging_table = dbutils.widgets.get("rs_staging_table")
client_table = dbutils.widgets.get("client_table")

In [0]:
%skip
source_table = "dev_silver_lakehouse.staging.bears_oblist_stg"
rs_table = "dev_silver_lakehouse.metric_layer.rs_oblist"
rs_staging_table = "dev_silver_lakehouse.metric_layer.rs_oblist_staging"
client_table = "prd_bronze_raw.mart_bkp.client"

In [0]:
spark.sql(f"""
TRUNCATE TABLE {rs_staging_table};
""")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW rs_oblist_src AS
SELECT 
  vc.source_system_key AS source_system_key,
  vc.reporting_week_ending_date_key AS reporting_week_ending_date_key,
  vc.office_key AS office_key,
  COALESCE(vc.client_key, Cld.ClientKey) AS client_key,
  vc.payer_key AS payer_key,
  vc.collector_name AS collector_name,
  vc.inv_no AS inv_no,
  vc.ar_0_90 AS ar_0_90,
  vc.ar_91_180 AS ar_91_180,
  vc.ar_181_270 AS ar_181_270,
  vc.ar_271_plus AS ar_271_plus,
  vc.payor_type_code AS payor_type_code
FROM {source_table} vc
LEFT JOIN {client_table} Cld
  ON vc.client_no = CAST(Cld.SourceSystemId AS STRING)
  AND Cld.SourceSystem = 'bears'
""")
)

In [0]:
spark.sql(f"""
INSERT INTO {rs_staging_table} (
    source_system_key,
    reporting_week_ending_date_key,
    office_key,
    client_key,
    payer_key,
    collector_name,
    inv_no,
    ar_0_90,
    ar_91_180,
    ar_181_270,
    ar_271_plus,
    payor_type_code,
    _load_timestamp
)
SELECT
    source_system_key,
    reporting_week_ending_date_key,
    office_key,
    client_key,
    payer_key,
    collector_name,
    inv_no,
    ar_0_90,
    ar_91_180,
    ar_181_270,
    ar_271_plus,
    payor_type_code,
    current_timestamp() AS _load_timestamp
FROM rs_oblist_src
""")

In [0]:
spark.sql(f"""
INSERT INTO {rs_table} (
    source_system_key,
    reporting_week_ending_date_key,
    office_key,
    client_key,
    payer_key,
    collector_name,
    inv_no,
    ar_0_90,
    ar_91_180,
    ar_181_270,
    ar_271_plus,
    payor_type_code,
    _load_timestamp
)
SELECT
    source_system_key,
    reporting_week_ending_date_key,
    office_key,
    client_key,
    payer_key,
    collector_name,
    inv_no,
    ar_0_90,
    ar_91_180,
    ar_181_270,
    ar_271_plus,
    payor_type_code,
    current_timestamp() AS _load_timestamp
FROM rs_oblist_src
""")